# 数据质量检查：入门版

现在只做三件事：**读懂一条数据、检查空问题和空回答、看看训练时会截掉多少内容。**

从上到下逐格运行，使用项目的 `minilora` Python 环境。这里只读取数据和本地分词器，原数据不会被修改。


## 1. 读取训练集

JSONL 文件的每一行是一条问答。下面把它读成一个列表 `data`。这一版先只看训练集。


In [1]:
import json
from pathlib import Path

project_dir = Path(r"D:\LLMProjects\MiniLoRA")
train_file = project_dir / "data" / "medical" / "train.jsonl"

with train_file.open(encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

print("训练样本数：", len(data))

训练样本数： 5000


## 2. 看懂一条样本

- `instruction`：任务要求。
- `input`：用户的问题。
- `output`：希望模型学习的参考回答。

先读这一条，看看回答是否回应了问题。想换一条，把 `data[0]` 中的数字改掉。


In [2]:
example = data[0]

print("任务要求：", example["instruction"])
print("问题：", example["input"])
print("参考回答：", example["output"])

任务要求： 请以谨慎、专业、易懂的方式回答下面的医疗健康问题。
问题： 尿毒症性心包炎的发病原因？
参考回答： 本病是慢性肾功能不全的严重并发症,可分为尿毒症心包炎和透析后心包炎。患者血小板功能和凝血功能障碍，纤溶活性降低，导致出血性心包炎或出血纤维性心包炎，增加心脏压塞的危险。患者如果患有甲状旁腺功能亢进，甲状腺激素大量分泌，钙盐增加，沉积心包。伴有的容量超负荷、高尿酸血症、低蛋白血症等也可导致心包炎。尿素氮等毒性物质所致包膜化学性炎症。营养不良、免疫功能低下，频发细菌、病毒感染易波及心包，诱发该病。


## 3. 检查空问题和空回答

没有具体问题或参考回答的记录需要单独查看。下面只把有问题、有回答的样本放进 `train_data_not_empty`，供后面的长度统计使用；文件本身不变。

`strip()` 用来去掉两端空白，避免把只有空格的内容当作正常文本。


In [3]:
train_data_not_empty = []
empty_count = 0

for item in data:
    question = item.get("input") or ""
    answer = item.get("output") or ""

    if question.strip() == "" or answer.strip() == "":
        empty_count += 1
    else:
        train_data_not_empty.append(item)

print("空问题或空回答的样本数：", empty_count)
print("用于长度统计的样本数：", len(train_data_not_empty))


空问题或空回答的样本数： 0
用于长度统计的样本数： 5000


## 4. 准备分词器

模型处理的是 **token**，所以要用 Qwen 的分词器统计长度，不能直接把汉字个数当作 token 数。

下面直接定义与训练脚本相同的 `build_messages` 函数，将问答数据转换为 system、user、assistant 三条消息，再加载本地 Qwen 分词器。代码可以独立运行，不会启动训练。


In [4]:
from pathlib import Path
from transformers import AutoTokenizer


def build_messages(example):
    """将一条问答数据转换为 system、user、assistant 三条消息。"""
    user_content = example["instruction"].strip()
    input_text = example.get("input", "").strip()

    if input_text:
        user_content = f"{user_content}\n\n问题：{input_text}"

    return [
        {
            "role": "system",
            "content": "你是一名谨慎的医疗健康问答助手。回答应清晰、专业，并提醒用户必要时及时就医。",
        },
        {
            "role": "user",
            "content": user_content,
        },
        {
            "role": "assistant",
            "content": example["output"].strip(),
        },
    ]


model_dir = Path(r"D:\LLMProjects\LLMModels\Qwen2.5-0.5B-Instruct")

tokenizer = AutoTokenizer.from_pretrained(model_dir)

max_lengths = [256, 512, 768, 1024]  # 一次比较四种长度上限。


## 5. 检查长度和截断

一条训练输入由两部分组成：

```text
提示词和问题（prompt） + 参考回答（answer）
```

总长度超过 `max_length` 时，当前训练代码会截掉尾部。如果 prompt 自己就占满上限，整段回答都会被截掉，这条样本就没有回答 token 可供学习。

下面一次比较 256、512、768、1024 四种长度上限，把结果保存在表格变量 `results_df` 中，并直接显示。平均总长度是截断前的长度，所以四行的平均值相同。参考回答末尾加结束符的方式与当前训练代码一致。


In [5]:
import pandas as pd

if not train_data_not_empty:
    raise ValueError("没有可统计的样本，请先检查数据。")

lengths = []
prompt_lengths = []

for item in train_data_not_empty:
    messages = build_messages(item)

    # 前两条消息是 system 和 user；最后一条是参考回答。
    prompt_ids = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    )["input_ids"]

    answer = messages[-1]["content"]
    answer_ids = tokenizer(
        answer + tokenizer.eos_token,
        add_special_tokens=False,
    )["input_ids"]

    lengths.append(len(prompt_ids) + len(answer_ids))
    prompt_lengths.append(len(prompt_ids))

# 保存每种长度上限的统计结果。
results = []
for max_length in max_lengths:
    # 条件成立时计为 1，sum 把这些 1 加起来，得到样本数。
    truncated_count = sum(length > max_length for length in lengths)
    no_answer_count = sum(length >= max_length for length in prompt_lengths)

    results.append({
        "max_length": max_length,
        "样本数": len(lengths),
        "平均总长度(token)": round(sum(lengths) / len(lengths), 1),
        "截断样本数": truncated_count,
        "截断比例": f"{truncated_count / len(lengths):.2%}",
        "回答全部截掉的样本数": no_answer_count,
    })

# DataFrame 是 pandas 的表格；放在单元格最后一行即可直接显示。
results_df = pd.DataFrame(results)
results_df

max_length,样本数,平均总长度(token),截断样本数,截断比例,回答全部截掉的样本数
256,5000,196.0,963,19.26%,1
512,5000,196.0,301,6.02%,0
768,5000,196.0,36,0.72%,0
1024,5000,196.0,19,0.38%,0


## 怎样理解结果

- **平均长度**：帮助你了解样本通常有多长，不能单凭平均值决定长度上限。
- **发生截断**：说明部分参考回答没有被完整用于训练；不代表这些数据本身有错。
- **回答被全部截掉**：说明这条样本无法提供回答部分的训练信号，需要调整长度或预处理。